In [4]:
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sergi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sergi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [5]:
# Cargar dataset
file_path = "dataset_transacciones_colombia.csv"
df = pd.read_csv(file_path)

In [6]:
# Tokenizar descripciones
df["tokens"] = df["Descripcion"].apply(lambda x: word_tokenize(str(x).lower()))


In [7]:
# Entrenar Word2Vec
w2v_model = Word2Vec(sentences=df["tokens"], vector_size=100, window=5, min_count=2, workers=4)

def text_to_vec(text):
    vectors = [w2v_model.wv[word] for word in text if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

# Convertir texto a vectores
df["vector"] = df["tokens"].apply(text_to_vec)
X = np.vstack(df["vector"].values)


In [8]:
# Mapear categorías a índices
categorias = df["Categoria"].unique()
categoria_to_idx = {cat: idx for idx, cat in enumerate(categorias)}
df["Categoria_ID"] = df["Categoria"].map(categoria_to_idx)
y = df["Categoria_ID"].values

In [9]:
# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Configuración de LightGBM con hiperparámetros optimizados
params = {
    "objective": "multiclass",
    "num_class": len(categorias),
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbosity": -1,
    "seed": 42
}

In [10]:
# Entrenamiento con validación cruzada
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

In [14]:
for train_idx, val_idx in kf.split(X_train, y_train):
    train_data = lgb.Dataset(X_train[train_idx], label=y_train[train_idx])
    val_data = lgb.Dataset(X_train[val_idx], label=y_train[val_idx], reference=train_data)
    
    model = lgb.train(
        params, 
        train_data, 
        num_boost_round=1000, 
        valid_sets=[val_data], 
        valid_names=["validation"],  # Especificamos un nombre para el set de validación
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],  # Se reemplaza `verbose_eval`
    )


    y_pred = model.predict(X_train[val_idx])
    y_pred_labels = [x.argmax() for x in y_pred]
    accuracy = accuracy_score(y_train[val_idx], y_pred_labels)
    accuracies.append(accuracy)
    print(f"Fold Accuracy: {accuracy:.4f}")

print(f"Mean CV Accuracy: {np.mean(accuracies):.4f}")

Training until validation scores don't improve for 50 rounds
[100]	validation's multi_logloss: 0.000217304
[200]	validation's multi_logloss: 4.06347e-08
Early stopping, best iteration is:
[238]	validation's multi_logloss: 1.26302e-08
Fold Accuracy: 1.0000
Training until validation scores don't improve for 50 rounds
[100]	validation's multi_logloss: 0.000217304
[200]	validation's multi_logloss: 4.06342e-08
Early stopping, best iteration is:
[236]	validation's multi_logloss: 1.26691e-08
Fold Accuracy: 1.0000
Training until validation scores don't improve for 50 rounds
[100]	validation's multi_logloss: 0.000217304
[200]	validation's multi_logloss: 4.06356e-08
Early stopping, best iteration is:
[237]	validation's multi_logloss: 1.2657e-08
Fold Accuracy: 1.0000
Training until validation scores don't improve for 50 rounds
[100]	validation's multi_logloss: 0.000217304
[200]	validation's multi_logloss: 4.06406e-08
Early stopping, best iteration is:
[237]	validation's multi_logloss: 1.2636e-08


In [15]:
# Evaluación final en test set
y_pred_test = model.predict(X_test)
y_pred_test_labels = [x.argmax() for x in y_pred_test]
print("Test Accuracy:", accuracy_score(y_test, y_pred_test_labels))

Test Accuracy: 1.0


In [16]:
# Guardar el modelo
model.save_model("lightgbm_transactions_model.txt")
w2v_model.save("word2vec_model.bin")
